# 1) Summarize diagrams
Here, we load all the RAG chunk JSON files and go through each one to find chunks where the type is "figure". For those, we send the figure’s caption or related content to the GPT-4.1-mini API to get a short summary of what the figure shows, things like the axes, comparisons, or main trends. Then we store that summary back into a new key called image_summary and save everything in the rag_chunks_image_summary folder. This way, all the figures are now represented as text, making it much easier to embed and search later without depending on image embedding models that usually fail to capture the meaning of complex research plots.

In [5]:
import os, json, base64, mimetypes, time
from pathlib import Path
from openai import OpenAI

In [3]:
from dotenv import load_dotenv
load_dotenv("keys.env") 
client = OpenAI()

SRC_DIR = Path("data/rag_chunks")
DST_DIR = Path("data/rag_chunks_image_summary")
DST_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
MODEL   = "gpt-4.1-mini"   # vision-capable 
PROMPT  = ("You are writing a scientific paper. "
           "Describe the figure precisely: axes/units if visible, key trends, "
           "comparisons, anomalies, and the main takeaway. Be specific.")

### Helper functions for API

In [5]:
def _data_url(img_path: Path) -> str:
    mime = mimetypes.guess_type(str(img_path))[0] or "image/jpeg"
    b64  = base64.b64encode(img_path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def summarize_image(img_path: Path) -> str:
    url = _data_url(img_path)
    resp = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": PROMPT},
                {"type": "input_image", "image_url": url},
            ],
        }],
        stream=False,
    )
    return resp.output_text.strip()

### Summarizing Figures Across JSON Chunks

In [6]:
def summarize_figures_in_json(src_json: Path, root: Path = Path(".")):
    data = json.loads(src_json.read_text())
    # support either list of chunks or {"chunks":[...]}
    chunks = data.get("chunks") if isinstance(data, dict) else data
    if not isinstance(chunks, list):
        raise ValueError(f"Unexpected JSON structure in {src_json}")

    for ch in chunks:
        try:
            if ch.get("type") == "figure":
                md = ch.setdefault("metadata", {})
                img_rel = md.get("image_path") or md.get("image")  # be tolerant
                if img_rel:
                    img_path = (root / img_rel).resolve()
                    if img_path.exists():
                        # simple retry for transient errors
                        for attempt in range(4):
                            try:
                                md["image_summary"] = summarize_image(img_path)
                                break
                            except Exception as e:
                                if attempt == 3: 
                                    md["image_summary"] = f"[error] {type(e).__name__}: {e}"
                                time.sleep(1.5 * (2 ** attempt))
                    else:
                        md["image_summary"] = f"[missing image at {img_rel}]"
        except Exception as e:
            ch.setdefault("metadata", {})["image_summary"] = f"[error] {type(e).__name__}: {e}"

    # write out preserving original shape
    out = {"chunks": chunks} if isinstance(data, dict) and "chunks" in data else chunks
    (DST_DIR / src_json.name).write_text(json.dumps(out, ensure_ascii=False, indent=2))


In [ ]:
for f in sorted(SRC_DIR.glob("*.json")):
    summarize_figures_in_json(f, root=Path("."))  

print("Done: ", DST_DIR)

# 2) Obtain Embeddings
We take the processed JSON chunks, now containing both textual content and image_summary fields, and generate embeddings. By converting all figure information into text, this step creates a unified vector representation for every chunk, making it easy to store, search, and retrieve using text-based similarity methods later in the RAG pipeline

In [1]:
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
SRC = Path("data/rag_chunks_image_summary")      
DST = Path("data/RAG_IS/rag_embeddings")               
DST.mkdir(parents=True, exist_ok=True)

### Embedding model
We’re using gte-large-en-v1.5 from Alibaba‑NLP (434 M parameters, supports up to 8,192 input tokens). It achieved an average MTEB score of ~65.39 on English tasks. This means it delivers strong retrieval/semantic-similarity performance in its size class, making it a solid fit for embedding our figure-summary text without over-investing in extremely large models.

In [7]:
MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"       
BATCH = 64

model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

### Helper functions
The build_text() helper function converts each chunk into a clean text block ready for embedding. For figures, it combines the original caption with the GPT-generated image_summary; for paragraphs, it includes the raw content. Section and page info are prepended as lightweight context tags so embeddings retain structural awareness during retrieval. The iter_chunks() helper function safely reads a JSON file and returns its list of chunks, handling multiple possible formats like plain lists, {"chunks": [...]}, or other wrappers such as {"data": [...]}

In [8]:
def build_text(ch):
    md = ch.get("metadata", {})
    sec = md.get("section", "")
    page = md.get("page", "")
    head = f"[SECTION] {sec} [PAGE] {page}".strip()

    if ch.get("type") == "figure":
        cap  = ch.get("content", "") or ""
        summ = md.get("image_summary", "") or ""
        return f"{head}\n[FIGURE]\nCaption: {cap}\nVisual summary: {summ}".strip()
    else:  # paragraph (default)
        body = ch.get("content", "") or ""
        return f"{head}\n[PARAGRAPH]\n{body}".strip()

In [9]:
def iter_chunks(path: Path):
    # robust JSON load
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data                           # already a list of chunks
    if isinstance(data, dict):
        if "chunks" in data and isinstance(data["chunks"], list):
            return data["chunks"]             # {"chunks": [...]}
        # tolerate other wrappers like {"data":[...]}
        for k in ("data", "items"):
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"Unexpected dict schema in {path}")
    raise ValueError(f"Unexpected JSON type {type(data)} in {path}")

### Normalizing Embeddings
This function encodes a batch of text chunks using the SentenceTransformer model and returns L2-normalized vectors, making them cosine-similarity ready. The normalization ensures compatibility with FAISS IndexFlatIP, enabling efficient and accurate similarity search later in the retrieval stage.

In [10]:
def embed_texts(texts):
    return model.encode(texts,
                        batch_size=BATCH,
                        normalize_embeddings=True,
                        convert_to_numpy=True,
                        show_progress_bar=False,
                    ).astype("float32")

In [ ]:
for src in tqdm(sorted(SRC.glob("*.json"))):
    # collect
    entries, texts = [], []
    for ch in iter_chunks(src):
        md = ch.get("metadata", {})
        tfe = build_text(ch)
        entries.append({"id": ch.get("id"),
                        "doc_id": src.stem.split(".")[0],
                        "type": ch.get("type"),
                        "page": md.get("page"),
                        "section": md.get("section"),
                        "text_for_embedding": tfe,
                        "image_path": md.get("image_path") if ch.get("type") == "figure" else None,
                        "image_summary": md.get("image_summary") if ch.get("type") == "figure" else None,
                        "model": MODEL_ID,
        })
        texts.append(tfe)

    # embed
    embs = embed_texts(texts)

    # write JSONL (one file per source JSON)
    out_path = DST / f"{src.stem}.jsonl"
    with out_path.open("w") as f:
        for e, v in zip(entries, embs):
            e["embedding"] = [float(x) for x in v.tolist()]
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("Saved embeddings to:", DST)

# 3) Retrival
In this stage, we use the precomputed embeddings to perform similarity search—typically via a FAISS index or a vector database. When a user query comes in, it’s embedded using the same model, and the system retrieves the most semantically similar chunks (including figure summaries). This allows the RAG pipeline to surface both textual and visual insights in response to natural-language questions.

In [ ]:
!pip uninstall -y faiss-cpu faiss numpy
!pip install "numpy<2"         # installs 1.26.4 on Py3.10
!pip install faiss-cpu==1.8.0.post1

In [ ]:
# dealing with incompatibilty issue between faiss and np
import numpy, faiss
print(numpy.__version__, faiss.__version__)

In [17]:
import os, ujson as json, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss, pickle

In [14]:
EMB_DIR = Path("data/RAG_IS/rag_embeddings")
INDEX_PATH = Path("data/RAG_IS/rag.index")
META_PATH  = INDEX_PATH.with_suffix(".meta.pkl")

MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"   # same as used to embed
DIM = 1024

### FAISS Index
Here, we check if a prebuilt FAISS index already exists. If it does, we load it directly for instant retrieval. If not, we build a new cosine-similarity index from the current embeddings, save it to disk, and reuse it for faster initialization in future runs.

In [19]:
if INDEX_PATH.exists():
    print("Index found — loading existing FAISS index...")
    index = faiss.read_index(str(INDEX_PATH))

    # try to load meta; if EOF, rebuild it from JSONL files
    meta = None
    if Path(META_PATH).exists() and Path(META_PATH).stat().st_size > 0:
        try:
            with open(META_PATH, "rb") as f:
                meta = pickle.load(f)
        except EOFError:
            print("Meta file is empty/corrupt — rebuilding metadata...")
    else:
        print("Meta file missing — rebuilding metadata...")

    if meta is None:
        meta = []
        for jf in sorted(EMB_DIR.glob("*.jsonl")):
            with jf.open("r", encoding="utf-8") as f:
                for line in f:
                    rec = json.loads(line)
                    meta.append({
                        "id": rec.get("id"),
                        "doc_id": rec.get("doc_id"),
                        "type": rec.get("type"),
                        "page": rec.get("page"),
                        "section": rec.get("section"),
                        "text": rec.get("text_for_embedding"),
                        "image_path": rec.get("image_path"),
                    })
        with open(META_PATH, "wb") as f:
            pickle.dump(meta, f, protocol=pickle.HIGHEST_PROTOCOL)

    if index.ntotal != len(meta):
        print(f"Warning: index count ({index.ntotal}) != meta rows ({len(meta)})")

    print("Loaded index with", index.ntotal, "vectors")

else:
    print("No existing index found — build your index path first.")

Index found — loading existing FAISS index...
Meta file missing — rebuilding metadata...
Loaded index with 5077 vectors


### Search & Display Results
These functions handle retrieval and inspection. The search() function embeds a text query and finds the top-k most similar chunks using the FAISS index. The show() function prints those results with scores, metadata, and short text previews.

In [20]:
# keep this outside search(), else it would reload on every query
model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

def search(query: str, k: int = 10):
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    sims, ids = index.search(q, k)
    return sims[0], ids[0]


def show(query: str, k: int = 10, max_chars: int = 500):
    sims, ids = search(query, k)
    print(f"\nQuery: {query}\nTop {k} results:\n")
    
    for rank, (sid, score) in enumerate(zip(ids, sims), 1):
        m = meta[int(sid)]
        snippet = (m["text"] or "")[:max_chars].replace("\n", " ")
        print(f"{rank:>2}. score={score:.4f}  [{m['type']}]  {m['doc_id']}  p.{m['page']}  {m['section']}")
        print(f"    {snippet}")
        if m["type"] == "figure" and m.get("image_path"):
            print(f"    (image: {m['image_path']})")
        print()

In [22]:
query = "training loss curves comparing AdamW vs SGD on CIFAR-10"
show(query, k=5, max_chars=500)


Query: training loss curves comparing AdamW vs SGD on CIFAR-10
Top 5 results:

 1. score=0.7345  [paragraph]  2203  p.9  Model and training details
    [SECTION] Model and training details [PAGE] 9 [PARAGRAPH] 8 Interestingly , a model trained with AdamW only passes the training performance of a model trained with Adam around 80% of the way through the cosine cycle, though the ending performance is notably better- see Figure A7

 2. score=0.7242  [paragraph]  2306  p.15  References
    [SECTION] References [PAGE] 15 [PARAGRAPH] Jingzhao Zhang, Sai Praneeth Karimireddy, Andreas Veit, Seungyeon Kim, Sashank J Reddi, Sanjiv Kumar, and Suvrit Sra. Why {adam} beats {sgd} for attention models, 2020.

 3. score=0.7030  [paragraph]  2203  p.9  Model and training details
    [SECTION] Model and training details [PAGE] 9 [PARAGRAPH] We use AdamW (Loshchilov and Hutter, 2019) for Chinchilla rather than Adam (Kingma and Ba, 2014) as this improves the language modelling loss and the downstream tas

# Generating Answers from Retrieved Context
In this stage, we take the top retrieved chunks and feed them into a language model (like LLaMA) along with the user’s query to generate a grounded response. The goal is not pure text generation but context-aware synthesis—the model uses retrieved passages and figure summaries to construct concise, factual answers.

In [23]:
import gc, torch, sys

# drop big globals if they exist
for name in ["model","tok","encoder","index","faiss_index"]:
    if name in globals(): del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [24]:
!nvidia-smi

Sun Nov  2 22:50:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             30W /   70W |     151MiB /  15360MiB |     76%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
#!kill -9 315051 870019 
# import subprocess, textwrap
# print(subprocess.getoutput("nvidia-smi"))

In [25]:
import torch, faiss, ujson as json
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [26]:
# Load FAISS index + metadata 
index = faiss.read_index(str(INDEX_PATH))
meta = []
for jf in sorted(EMB_DIR.glob("*.jsonl")):
    with jf.open() as f:
        for line in f:
            meta.append(json.loads(line))
print(f"Loaded {len(meta)} chunks from {len(list(EMB_DIR.glob('*.jsonl')))} files.")

Loaded 5077 chunks from 23 files.


In [27]:
# Load embedding model for query encoding
encoder = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", trust_remote_code=True)

In [28]:
def retrieve(query, k=10):
    q_vec = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q_vec, k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        item = meta[int(i)]
        results.append({"score": float(score), **item})
    return results

### Generator Model
Here, we load LLaMA 3.1 8B Instruct, a strong open-source instruction-tuned model from Meta. To make inference efficient on limited GPUs, we enable 4-bit quantization using BitsAndBytesConfig with the NF4 scheme and mixed-precision (float16) compute. This setup keeps the model lightweight while maintaining most of its reasoning and generation quality

In [29]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading checkpoint shards: 100%|██████████| 4/4 [01:24<00:00, 21.24s/it]


In [30]:
!nvidia-smi

Sun Nov  2 22:54:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             29W /   70W |    8857MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


### System Behavior and Generation Settings
This section sets global configurations for the generator.
The SYSTEM prompt defines how the model should behave, restricting it to use only the provided <chunk> context, paraphrasing instead of copying, and citing sources precisely.
The GEN_CFG dictionary holds decoding parameters such as token limit, temperature, and repetition controls, ensuring responses stay factual, concise, and non-repetitive across all queries.

In [72]:
SYSTEM = (
  "Answer ONLY from <chunk> context. Read EVERY chunk.\n"
  "Step 1: Extract EVERY distinct method/model by its PROPER NAME as written "
  "(use exact capitalization), then give a 1-line description.\n"
  "Step 2: Write a concise answer that covers each named item.\n"
  "If nothing relevant: Not found in the given context.\n"
  "Paraphrase descriptions; DO NOT rename methods. Cite as [DOC:doc_id, p:page]."
)

GEN_CFG = dict(
    max_new_tokens=512,        # maximum tokens model can generate in the output
    temperature=0.5,           # randomness; lower = more deterministic, higher = more creative
    top_p=0.9,                 # nucleus sampling; model samples only from top 90% probability mass
    do_sample=False,            # if True, enables stochastic sampling (else it picks argmax each time)
    repetition_penalty=1.01,   # penalizes repeating same tokens; >1 discourages loops/redundancy
    no_repeat_ngram_size=8,    # forbids repeating any 8-token sequence exactly
)

### Building Context
The SYSTEM prompt guides the model by defining its role, tone, and behavior, ensuring its responses are consistent, relevant, and aligned with specific goals. build_context() then prepares retrieved chunks for the generator by wrapping each one in a structured <chunk> block containing metadata such as doc_id, page, and type. It trims long text to a maximum length (max_chars) to fit within the model’s input window.

In [73]:
def build_context(chunks, max_chars=1000):
    blocks = []
    for c in chunks:
        text = (c.get("text_for_embedding") or c.get("text") or "")[:max_chars]
        blocks.append(
            f"<chunk doc_id='{c.get('doc_id')}' page='{c.get('page')}' type='{c.get('type')}'>\n{text}\n</chunk>"
        )
    return "\n".join(blocks)

### Generating Context-Grounded Answers
generate_answer() builds the final prompt by combining the system instructions, user query, and retrieved context. It tokenizes the final prompt, runs the model with tuned decoding parameters (temperature, top-p, repetition penalty, etc.), and decodes only the new tokens generated after the prompt. The output is a clean, research-grounded answer that paraphrases information and cites sources.

In [74]:
def generate_answer(query, context, max_new_tokens=GEN_CFG["max_new_tokens"]):
    if not context.strip():
        return "Not found in the given context."
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM}\n"
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"Question: {query}\n\n"
        "Use the <chunk> blocks below; paraphrase and cite as [DOC:doc_id, p:page].\n\n"
        f"{context}\n"
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        **{k:v for k,v in GEN_CFG.items() if k != "max_new_tokens"},
        max_new_tokens=max_new_tokens,
        eos_token_id=[tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")],
        pad_token_id=tok.eos_token_id,
    )
    gen_ids = out[:, inputs["input_ids"].shape[1]:]
    return tok.decode(gen_ids[0], skip_special_tokens=True).strip()

In [77]:
query = "What methods exist to accelerate token generation during inference without retraining the language model?"
chunks = retrieve(query, k=10)
context = build_context(chunks, max_chars=1000) 
answer = generate_answer(query, context, max_new_tokens=512)

In [78]:
print(answer)

Here are the distinct methods/models extracted from the given context:

1. **Speculative Decoding**: A technique that uses a draft model for token generation and verifies those tokens using a larger model.
2. **Token Dropping**: A method that skips multiple tokens at a given time step in recurrent neural networks.
3. **KV Cache Compression**: A technique that compresses the input sequence by adding polling layers to the encoding modules of the Transformer architecture.
4. **Token Selection**: A task that learns to select performance-crucious tokens in the original BERT model.
5. **Learnable Threshold**: A design that detects unimportant tokens to prune in the BERT model.
6. **Speculative Inference**: An algorithm that samples from autoregressive models faster without changing the outputs by computing several tokens in parallel.
7. **MEDUSA**: A method that enhances LLM inference speed by equipping models with additional predictive decode heads, allowing for generating multiple tokens c

In [66]:
for i, ch in enumerate(chunks, 1):
    print(f"--- Chunk {i} | Score: {ch['score']:.3f} | Doc: {ch['doc_id']} | Page: {ch.get('page')} ---")
    print(ch.get("text_for_embedding", ch.get("text", ""))[:800])  # limit to 800 chars for readability
    print()

--- Chunk 1 | Score: 0.762 | Doc: 2312 | Page: 20 ---
[SECTION] Efficient Inference for Large Language Models. [PAGE] 20
[PARAGRAPH]
Speculative Execution. Speculative decoding (Leviathan et al., 2022; Zhang et al., 2023b; He et al., 2023) is a technique that uses a draft model for generation and uses the larger model to verify those tokens. This technique is orthogonal to us and can be used for further improvement. In the case of speculative decoding, the window in our method is updated with multiple tokens rather than one.

--- Chunk 2 | Score: 0.750 | Doc: 2401 | Page: 10 ---
[SECTION] REFERENCES [PAGE] 10
[PARAGRAPH]
Sebastian Borgeaud, Arthur Mensch, Jordan Hoffmann, Trevor Cai, Eliza Rutherford, Katie Millican, George Bm Van Den Driessche, Jean-Baptiste Lespiau, Bogdan Damoc, Aidan Clark, et al. Improving language models by retrieving from trillions of tokens. In International conference on machine learning , pp. 2206-2240. PMLR, 2022. URL https://arxiv.org/abs/2112. 04426 .

---